In [1]:
bbs = []
for path in tqdm(labels):
    with open(path) as f:
        lines = f.readlines()
    bbs.append(lines[-1][:-1].split(' '))

NameError: name 'tqdm' is not defined

In [ ]:
df['bb'] = bbs

In [ ]:
lbs = [lb.split('/')[4] for lb in labels]

In [ ]:
df['label'] = lbs

In [ ]:
df.info()

In [ ]:
df.to_csv('cars.csv')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision
from torchvision import transforms

In [ ]:
class CustomDataset(Dataset):
    
    def __init__(self, df):
        self.images = df['img_path'].values
        self.bbs = df['bb'].values
        self.labels = df['label'].values
        
    def __getitem__(self, index):
        image_path = self.images[index]
        try:
            image = Image.open(image_path)
        except:
            image = Image.open(self.images[0])
            print('OPEN FILE ERR', image_path)
        left, top, right, bottom = self.bbs[index]
        try:
            image = image.crop((int(left), int(top), int(right), int(bottom)))
        except:
            print('BB ERRR', image_path)
        image = image.resize((224, 224))
        image = transforms.ToTensor()(image)
        try:
            image = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(image)
        except:
            image = torch.zeros([3, 224, 224])
            print('IMAGE FORMAT ERRR', image_path)
        if image.shape != torch.Size([3, 224, 224]):
            image = torch.zeros([3, 224, 224])
            print('IMG ERR', image_path)
        label = int(self.labels[index]) - 1
        return image, label
        
    def __len__ (self):
        return len(self.images)

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)

dataset_train = CustomDataset(train)
dataset_test = CustomDataset(test)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
from torchvision import models
model = models.resnet101(pretrained=True)
n_classes = len(list(set(df['label'].values)))
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, n_classes)
model.to(device)
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=10, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=2, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
def train_nn(model, data_loader):
    loss_fn = torch.nn.CrossEntropyLoss()
    model.train()
    for data in tqdm(data_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
def eval_nn(model, data_loader):
    predicted = []
    labels = []
    model.eval()
    with torch.no_grad():
        for data in tqdm(data_loader):
            x, y = data
            x = x.to(device)

            outputs = model(x)
            _, predict = torch.max(outputs.data, 1)
            predict = predict.cpu().detach().numpy().tolist()
            predicted += predict
            labels += y
        print(f1_score(labels, predicted, average='macro'))
    return labels, predicted
for epoch in range(1):
    train_nn(model, train_loader)
    eval_nn(model, train_loader)
    eval_nn(model, test_loader)
for epoch in range(10):
    train_nn(model, train_loader)
    eval_nn(model, train_loader)
    eval_nn(model, test_loader)